<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/media/update_alt_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✏️ WordPress Alt Text Updater — v1

**Description:** Script untuk mengupdate alt text gambar di WordPress Media Library secara massal berdasarkan file CSV/Excel, dengan opsi diagnosa koneksi sebelum menjalankan update.

---

## ✅ Expected Result

| Field | Sebelum | Sesudah |
|---|---|---|
| `id` | `1042` | `1042` |
| `url_filename` | `bali-sunset-uluwatu.jpg` | `bali-sunset-uluwatu.jpg` |
| `alt_text` (di WP) | *(kosong)* | `bali sunset uluwatu \| infojatengpos.com` |

Format konversi alt text:

| url_filename | → Alt Text |
|---|---|
| `realme-perkenalkan-hp-baru.webp` | `realme perkenalkan hp baru \| infojatengpos.com` |
| `bali_sunset_2024.jpg` | `bali sunset 2024 \| infojatengpos.com` |

### Kolom CSV/Excel yang dibutuhkan

| Kolom | Wajib? | Keterangan |
|---|---|---|
| `id` | ✅ | ID media WordPress |
| `url_filename` | ✅ | Nama file gambar |

---

## ⚙️ Cara Pakai

| Langkah | Perintah |
|---|---|
| 1. Diagnosa dulu | `diagnose(test_id=7842)` |
| 2. Jika ada redirect, ganti `BASE_URL` | `BASE_URL = "https://infojatengpos.com"` |
| 3. Jalankan update | `main()` |

---

## 🔗 Links

| Resource | URL |
|---|---|
| 📓 Google Colab | [Buka Notebook](https://colab.research.google.com/drive/1c9EpzAaGXITqXICvmQCaSnREAFHCp4mK?usp=sharing) |
| 🤖 Claude Chat | [Buka Percakapan](https://claude.ai/chat/c21ae102-d05b-41f5-9e6e-d654bb425a13) |
| 📖 WP REST API Docs | [developer.wordpress.org/rest-api](https://developer.wordpress.org/rest-api/reference/media/) |

---


In [ ]:
# @title
"""
Script untuk update Alt Text gambar WordPress
berdasarkan kolom url_filename dari file CSV/Excel yang diupload.

Format alt text:
  "realme-perkenalkan-lumacolor.webp" -> "realme perkenalkan lumacolor | infojatengpos.com"

Kredential diambil dari Google Colab Secrets:
  - syahid_infojateng      -> username
  - syahid_infojatengpass  -> password
"""

import requests
import os
import re
import time
from pathlib import Path

# ============================================================
# RICH LOGGER SETUP
# ============================================================
try:
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn, TaskProgressColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich.text import Text
    from rich import box
    from rich.rule import Rule
    from rich.prompt import Prompt, Confirm
    from rich.columns import Columns
    from rich.live import Live
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rich", "-q"])
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn, TaskProgressColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich.text import Text
    from rich import box
    from rich.rule import Rule
    from rich.prompt import Prompt, Confirm
    from rich.columns import Columns
    from rich.live import Live

console = Console()

def log_ok(msg):     console.print(f"[bold green]  ✔[/bold green]  {msg}")
def log_fail(msg):   console.print(f"[bold red]  ✘[/bold red]  {msg}")
def log_skip(msg):   console.print(f"[bold yellow]  ⊘[/bold yellow]  {msg}")
def log_info(msg):   console.print(f"[bold cyan]  ℹ[/bold cyan]  {msg}")
def log_warn(msg):   console.print(f"[bold yellow]  ⚠[/bold yellow]  {msg}")
def log_error(msg):  console.print(f"[bold red]  ✘[/bold red]  [red]{msg}[/red]")

# ============================================================
# AMBIL KREDENTIAL DARI GOOGLE COLAB SECRETS
# ============================================================
try:
    from google.colab import userdata
    WP_PASSWORD = userdata.get('syahid_infojatengpass')
    WP_USERNAME = userdata.get('syahid_infojateng')
    log_ok("Kredential berhasil diambil dari Google Colab Secrets")
except Exception as e:
    log_warn(f"Gagal ambil dari Colab Secrets: {e}")
    WP_USERNAME = ""
    WP_PASSWORD = ""

# ============================================================
# KONFIGURASI
# ============================================================
BASE_URL   = "https://www.balitouristic.com/"
API_URL    = f"{BASE_URL}/wp-json/wp/v2/media"
DOMAIN     = "balitouristic.com"
DELAY_SEC  = 0.3

# ============================================================
# DIAGNOSA
# ============================================================

def diagnose(test_id=None):
    """
    Cek koneksi, auth, permission, dan redirect.
    Contoh pakai: diagnose(test_id=7842)
    """
    session = requests.Session()
    session.auth = (WP_USERNAME, WP_PASSWORD)

    console.print()
    console.print(Panel(
        f"[bold]BASE URL :[/bold] {BASE_URL}\n"
        f"[bold]User     :[/bold] {WP_USERNAME}\n"
        f"[bold]Test ID  :[/bold] {test_id or '[dim]tidak diset[/dim]'}",
        title="[bold blue]🔬 Diagnosa Koneksi & Auth",
        border_style="blue",
        padding=(0, 2),
    ))

    results = []

    # 1. Redirect check
    try:
        r = requests.get(BASE_URL, timeout=10, allow_redirects=False)
        if r.status_code in (301, 302):
            loc = r.headers.get('Location', '').rstrip('/')
            results.append(("[red]REDIRECT[/red]", "⚠ MASALAH", f"Redirect ke [bold]{loc}[/bold] — auth hilang! Ganti BASE_URL"))
            # Test HTTPS langsung
            s2 = requests.Session()
            s2.auth = (WP_USERNAME, WP_PASSWORD)
            r2 = s2.get(f"{loc}/wp-json/wp/v2/users/me", timeout=15)
            if r2.status_code == 200:
                me = r2.json()
                results.append(("[cyan]HTTPS TEST[/cyan]", "[green]OK[/green]", f"Login OK sebagai [bold]{me.get('name')}[/bold] di HTTPS"))
            else:
                results.append(("[cyan]HTTPS TEST[/cyan]", f"[red]HTTP {r2.status_code}[/red]", "Cek BASE_URL dan credentials"))
        else:
            results.append(("[cyan]REDIRECT[/cyan]", "[green]OK[/green]", f"Tidak ada redirect (HTTP {r.status_code})"))
    except Exception as e:
        results.append(("[cyan]REDIRECT[/cyan]", "[red]ERROR[/red]", str(e)))

    # 2. REST API
    try:
        r = session.get(f"{BASE_URL}/wp-json/wp/v2", timeout=15)
        status = "[green]OK[/green]" if r.status_code == 200 else f"[red]HTTP {r.status_code}[/red]"
        results.append(("[cyan]REST API[/cyan]", status, f"HTTP {r.status_code}"))
    except Exception as e:
        results.append(("[cyan]REST API[/cyan]", "[red]ERROR[/red]", str(e)))

    # 3. Auth login
    try:
        r = session.get(f"{BASE_URL}/wp-json/wp/v2/users/me", timeout=15)
        if r.status_code == 200:
            me = r.json()
            results.append(("[cyan]AUTH[/cyan]", "[green]OK[/green]", f"Login sebagai [bold]{me.get('name')}[/bold] | role: {me.get('roles', [])}"))
        elif r.status_code == 401:
            results.append(("[cyan]AUTH[/cyan]", "[red]401 GAGAL[/red]", "Cek username/password — gunakan Application Password"))
        else:
            results.append(("[cyan]AUTH[/cyan]", f"[red]HTTP {r.status_code}[/red]", r.text[:120]))
    except Exception as e:
        results.append(("[cyan]AUTH[/cyan]", "[red]ERROR[/red]", str(e)))

    # 4. Test update
    if test_id:
        try:
            url = f"{API_URL}/{test_id}"
            r = session.post(url, json={"alt_text": "test diagnostik"}, timeout=15)
            if r.status_code == 200:
                results.append(("[cyan]UPDATE[/cyan]", "[bold green]✔ SIAP[/bold green]", f"ID {test_id} berhasil diedit — semua siap!"))
                session.post(url, json={"alt_text": ""}, timeout=15)
            elif r.status_code == 401:
                results.append(("[cyan]UPDATE[/cyan]", "[red]401 GAGAL[/red]", "Tidak terautentikasi untuk edit"))
            elif r.status_code == 403:
                results.append(("[cyan]UPDATE[/cyan]", "[red]403 GAGAL[/red]", "Tidak punya izin edit media"))
            else:
                try:
                    err = r.json()
                    results.append(("[cyan]UPDATE[/cyan]", f"[red]HTTP {r.status_code}[/red]", f"{err.get('code','?')} — {err.get('message','?')}"))
                except Exception:
                    results.append(("[cyan]UPDATE[/cyan]", f"[red]HTTP {r.status_code}[/red]", r.text[:120]))
        except Exception as e:
            results.append(("[cyan]UPDATE[/cyan]", "[red]ERROR[/red]", str(e)))

    # Render tabel diagnosa
    table = Table(box=box.ROUNDED, border_style="dim", header_style="bold magenta", show_lines=True)
    table.add_column("Cek",     style="bold", min_width=12)
    table.add_column("Status",  justify="center", min_width=14)
    table.add_column("Detail",  min_width=50)

    for check, status, detail in results:
        table.add_row(check, status, detail)

    console.print(table)
    console.print()


# ============================================================
# FUNGSI HELPER
# ============================================================

def filename_to_alt(filename):
    name = re.sub(r'\.[^.]+$', '', filename)
    name = name.replace('-', ' ').replace('_', ' ')
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return f"{name} | {DOMAIN}"


def load_file(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == '.csv':
        import csv
        with open(filepath, 'r', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
    elif ext in ('.xlsx', '.xls'):
        import pandas as pd
        rows = pd.read_excel(filepath).to_dict(orient='records')
    else:
        raise ValueError(f"Format tidak didukung: {ext}. Gunakan .csv atau .xlsx")
    return rows


def find_column(rows, target):
    cols = list(rows[0].keys())
    for col in cols:
        if col.strip().lower() == target.lower():
            return col
    raise ValueError(f"Kolom '{target}' tidak ditemukan. Tersedia: {cols}")


def update_alt_text(session, media_id, alt_text):
    url = f"{API_URL}/{media_id}"
    try:
        resp = session.post(url, json={"alt_text": alt_text}, timeout=30)
        if resp.status_code == 200:
            return True
        try:
            err = resp.json()
            log_fail(f"HTTP {resp.status_code} | {err.get('code','?')} — {err.get('message','?')}")
        except Exception:
            log_fail(f"HTTP {resp.status_code} | {resp.text[:150]}")
        return False
    except Exception as e:
        log_fail(f"Exception: {e}")
        return False


def upload_file_colab():
    try:
        from google.colab import files
        log_info("Silakan upload file CSV atau Excel...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Tidak ada file yang diupload.")
        filename = list(uploaded.keys())[0]
        log_ok(f"File [bold]{filename}[/bold] berhasil diupload.")
        return filename
    except ImportError:
        path = Prompt.ask("[cyan]Masukkan path file CSV/Excel").strip()
        if not os.path.exists(path):
            raise FileNotFoundError(f"File tidak ditemukan: {path}")
        return path


# ============================================================
# MAIN
# ============================================================

def main():
    console.print()
    console.rule("[bold magenta]🖼  WordPress Alt Text Updater[/bold magenta]")

    if not WP_USERNAME or not WP_PASSWORD:
        console.print(Panel(
            "[bold red]Username / Password kosong![/bold red]\n\n"
            "Pastikan Google Colab Secrets sudah diset:\n"
            "  • [cyan]syahid_infojateng[/cyan]      → username\n"
            "  • [cyan]syahid_infojatengpass[/cyan]  → password",
            title="[bold red]❌ Error",
            border_style="red",
        ))
        return

    filepath = upload_file_colab()

    log_info(f"Membaca file: [bold]{filepath}[/bold]")
    rows = load_file(filepath)

    col_filename = find_column(rows, 'url_filename')
    col_id = None
    try:
        col_id = find_column(rows, 'id')
        log_ok(f"Kolom ID ditemukan: [bold]{col_id}[/bold]")
    except ValueError:
        log_warn("Kolom 'id' tidak ada — akan cari via filename (lebih lambat).")

    console.print(Panel(
        f"[bold]File          :[/bold] {filepath}\n"
        f"[bold]Kolom filename :[/bold] {col_filename}\n"
        f"[bold]Kolom ID      :[/bold] {col_id or '[dim]tidak ditemukan[/dim]'}\n"
        f"[bold]Total baris   :[/bold] [green]{len(rows)}[/green]",
        title="[bold blue]📂 Info File",
        border_style="blue",
        padding=(0, 2),
    ))

    # Preview tabel konversi
    console.print()
    console.print("[bold]Preview konversi (5 baris pertama):[/bold]")
    prev_table = Table(box=box.SIMPLE_HEAVY, header_style="bold magenta", show_lines=False)
    prev_table.add_column("url_filename",  style="cyan",  max_width=45)
    prev_table.add_column("→ Alt Text",    style="green", max_width=60)

    for row in rows[:5]:
        fn = str(row.get(col_filename, '')).strip()
        prev_table.add_row(fn, filename_to_alt(fn))
    console.print(prev_table)

    # Confirm
    console.print()
    confirm = Confirm.ask("[bold yellow]Lanjutkan update ke WordPress?[/bold yellow]")
    if not confirm:
        log_warn("Dibatalkan oleh user.")
        return

    console.print()
    session = requests.Session()
    session.auth = (WP_USERNAME, WP_PASSWORD)

    sukses, gagal, skip = 0, 0, 0
    total = len(rows)

    with Progress(
        SpinnerColumn(style="bold magenta"),
        TextColumn("[bold blue]{task.description}"),
        BarColumn(bar_width=35, style="magenta", complete_style="bold green"),
        TaskProgressColumn(),
        MofNCompleteColumn(),
        TimeElapsedColumn(),
        console=console,
    ) as progress:
        task = progress.add_task("Updating...", total=total)

        for i, row in enumerate(rows, 1):
            fn = str(row.get(col_filename, '')).strip()

            if not fn:
                skip += 1
                progress.advance(task)
                continue

            alt_text = filename_to_alt(fn)

            media_id = None
            if col_id and row.get(col_id):
                try:
                    media_id = int(float(str(row[col_id])))
                except Exception:
                    media_id = None

            if not media_id:
                log_skip(f"[{i}/{total}] Tidak ada ID untuk: [dim]{fn}[/dim]")
                skip += 1
                progress.advance(task)
                continue

            ok = update_alt_text(session, media_id, alt_text)

            if ok:
                sukses += 1
                progress.update(
                    task,
                    advance=1,
                    description=f"[green]✔ OK[/green]  ID [bold]{media_id}[/bold] | {fn[:35]}",
                )
            else:
                gagal += 1
                progress.update(
                    task,
                    advance=1,
                    description=f"[red]✘ GAGAL[/red] ID [bold]{media_id}[/bold] | {fn[:35]}",
                )

            time.sleep(DELAY_SEC)

    # Summary panel
    console.print()
    console.print(Panel(
        f"[bold green]✔ Berhasil  : {sukses}[/bold green]\n"
        f"[bold red]✘ Gagal     : {gagal}[/bold red]\n"
        f"[bold yellow]⊘ Skip      : {skip}[/bold yellow]\n"
        f"─────────────────\n"
        f"[bold]Total       : {sukses + gagal + skip}[/bold]",
        title="[bold]📊 Hasil Update",
        border_style="magenta" if gagal == 0 else "yellow",
        padding=(0, 2),
    ))


# ============================================================
# CARA PAKAI DI COLAB:
#
# Langkah 1 - Jalankan diagnosa dulu (ganti 7842 dengan ID dari file kamu):
#   diagnose(test_id=7842)
#
# Langkah 2 - Jika ada redirect, ganti BASE_URL di atas (misal ke https://)
#   Lalu jalankan ulang cell import
#
# Langkah 3 - Jika diagnosa UPDATE: SIAP, jalankan:
#   main()
# ============================================================

  ✔  Kredential berhasil diambil dari Google Colab Secrets

In [ ]:
# Cell 2 - Diagnosa dulu
diagnose(test_id=7842)

In [ ]:
# Cell 3 - Update massal
main()

---

# ✏️ WordPress Alt Text Updater — v2 (Auto Domain)

**Description:** Versi ringkas dari v1 — tinggal upload CSV hasil `wp_media_fetcher.py`, domain dan API URL otomatis terdeteksi, langsung update tanpa konfigurasi manual.

---

## ✅ Expected Result

| Field | Sebelum | Sesudah |
|---|---|---|
| `id` | `1042` | `1042` |
| `url_filename` | `bali-sunset-uluwatu.jpg` | `bali-sunset-uluwatu.jpg` |
| `url_netloc` | `www.balitouristic.com` | `www.balitouristic.com` |
| `alt_text` (di WP) | *(kosong)* | `bali sunset uluwatu \| balitouristic.com` |

Format konversi alt text:

| url_filename | url_netloc | → Alt Text |
|---|---|---|
| `bali-sunset-uluwatu.jpg` | `www.balitouristic.com` | `bali sunset uluwatu \| balitouristic.com` |
| `wisata-kuta-beach.webp` | `www.balitouristic.com` | `wisata kuta beach \| balitouristic.com` |

### Kolom CSV yang dibutuhkan

| Kolom | Wajib? | Keterangan |
|---|---|---|
| `id` | ✅ | ID media WordPress |
| `url_filename` | ✅ | Nama file gambar |
| `url_netloc` | ✅ | Domain — dipakai otomatis untuk alt text & API URL |

> ✅ Semua kolom ini sudah tersedia di output `wp_media_fetcher.py`

---

## ⚙️ Cara Pakai

| Langkah | Keterangan |
|---|---|
| 1. Run cell | Script langsung jalan |
| 2. Upload CSV | Pilih file `wp_media_images.csv` |
| 3. Cek preview | Pastikan konversi alt text sudah benar |
| 4. Konfirmasi | Ketik Y → update berjalan |

---

## 🔗 Links

| Resource | URL |
|---|---|
| 📓 Google Colab | [Buka Notebook](https://colab.research.google.com/drive/1c9EpzAaGXITqXICvmQCaSnREAFHCp4mK?usp=sharing) |
| 🤖 Claude Chat | [Buka Percakapan](https://claude.ai/chat/c21ae102-d05b-41f5-9e6e-d654bb425a13) |
| 📖 WP REST API Docs | [developer.wordpress.org/rest-api](https://developer.wordpress.org/rest-api/reference/media/) |
| 📄 Script Fetcher | `wp_media_fetcher.py` |

In [ ]:
# @title
"""
Script untuk update Alt Text gambar WordPress — Versi 2
Tinggal upload CSV hasil wp_media_fetcher.py, langsung jalan.

- Domain otomatis diambil dari kolom url_netloc di CSV
- Alt text dibentuk dari url_filename
- Format: "bali-sunset.jpg" -> "bali sunset | balitouristic.com"

Kredential diambil dari Google Colab Secrets.
"""

import requests
import re
import time
from pathlib import Path

# ============================================================
# RICH LOGGER SETUP
# ============================================================
try:
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn, TaskProgressColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    from rich.prompt import Confirm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rich", "-q"])
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn, TaskProgressColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    from rich.prompt import Confirm

console = Console()

def log_ok(msg):    console.print(f"[bold green]  ✔[/bold green]  {msg}")
def log_fail(msg):  console.print(f"[bold red]  ✘[/bold red]  {msg}")
def log_skip(msg):  console.print(f"[bold yellow]  ⊘[/bold yellow]  {msg}")
def log_info(msg):  console.print(f"[bold cyan]  ℹ[/bold cyan]  {msg}")
def log_warn(msg):  console.print(f"[bold yellow]  ⚠[/bold yellow]  {msg}")

# ============================================================
# AMBIL KREDENTIAL DARI GOOGLE COLAB SECRETS
# ============================================================
try:
    from google.colab import userdata
    WP_PASSWORD = userdata.get('pass_bali')
    WP_USERNAME = userdata.get('admin_bali')
    log_ok("Kredential berhasil diambil dari Google Colab Secrets")
except Exception as e:
    log_warn(f"Gagal ambil dari Colab Secrets: {e}")
    WP_USERNAME = ""
    WP_PASSWORD = ""

DELAY_SEC = 0.3

# ============================================================
# HELPER
# ============================================================

def filename_to_alt(filename, domain):
    """'bali-sunset-uluwatu.webp' -> 'bali sunset uluwatu | balitouristic.com'"""
    name = re.sub(r'\.[^.]+$', '', filename)
    name = name.replace('-', ' ').replace('_', ' ')
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return f"{name} | {domain}"


def load_csv(filepath):
    import csv
    with open(filepath, 'r', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def upload_csv():
    try:
        from google.colab import files
        log_info("Silakan upload file CSV hasil [bold]wp_media_fetcher.py[/bold]...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Tidak ada file yang diupload.")
        filename = list(uploaded.keys())[0]
        log_ok(f"File [bold]{filename}[/bold] berhasil diupload.")
        return filename
    except ImportError:
        from rich.prompt import Prompt
        path = Prompt.ask("[cyan]Masukkan path file CSV").strip()
        if not Path(path).exists():
            raise FileNotFoundError(f"File tidak ditemukan: {path}")
        return path


def update_alt_text(session, api_url, media_id, alt_text):
    try:
        resp = session.post(f"{api_url}/{media_id}", json={"alt_text": alt_text}, timeout=30)
        if resp.status_code == 200:
            return True
        try:
            err = resp.json()
            log_fail(f"HTTP {resp.status_code} | {err.get('code','?')} — {err.get('message','?')}")
        except Exception:
            log_fail(f"HTTP {resp.status_code} | {resp.text[:120]}")
        return False
    except Exception as e:
        log_fail(f"Exception: {e}")
        return False


# ============================================================
# MAIN
# ============================================================

def main():
    console.print()
    console.rule("[bold magenta]🖼  WordPress Alt Text Updater  [dim]v2 — Auto Domain[/dim][/bold magenta]")
    console.print()

    if not WP_USERNAME or not WP_PASSWORD:
        console.print(Panel(
            "[bold red]Username / Password kosong![/bold red]\n\n"
            "Pastikan Google Colab Secrets sudah diset:\n"
            "  • [cyan]pass_bali[/cyan]   → password\n"
            "  • [cyan]admin_bali[/cyan]  → username",
            title="[bold red]❌ Error", border_style="red",
        ))
        return

    # Upload & baca CSV
    filepath = upload_csv()
    rows = load_csv(filepath)

    if not rows:
        log_warn("File CSV kosong.")
        return

    # Validasi kolom wajib
    required = ['id', 'url_filename', 'url_netloc']
    missing = [c for c in required if c not in rows[0]]
    if missing:
        console.print(Panel(
            f"[red]Kolom berikut tidak ditemukan di CSV:[/red]\n"
            + "\n".join(f"  • [bold]{c}[/bold]" for c in missing)
            + f"\n\n[dim]Kolom tersedia: {', '.join(rows[0].keys())}[/dim]",
            title="[bold red]❌ Kolom Tidak Lengkap", border_style="red",
        ))
        return

    # Ambil domain dari baris pertama yang valid
    domain = next(
        (str(r.get('url_netloc', '')).strip() for r in rows if r.get('url_netloc')),
        "unknown.com"
    )

    # Bangun API URL dari domain
    api_url = f"https://{domain}/wp-json/wp/v2/media"

    # Filter baris valid (punya id & url_filename)
    valid_rows = [
        r for r in rows
        if str(r.get('id', '')).strip() and str(r.get('url_filename', '')).strip()
    ]
    skip_count = len(rows) - len(valid_rows)

    # Info panel
    console.print(Panel(
        f"[bold]File        :[/bold] {filepath}\n"
        f"[bold]Domain      :[/bold] [green]{domain}[/green]\n"
        f"[bold]API URL     :[/bold] {api_url}\n"
        f"[bold]Total baris :[/bold] {len(rows)}\n"
        f"[bold]Baris valid :[/bold] [green]{len(valid_rows)}[/green]\n"
        f"[bold]Skip (kosong):[/bold] [yellow]{skip_count}[/yellow]",
        title="[bold blue]📂 Konfigurasi Otomatis",
        border_style="blue",
        padding=(0, 2),
    ))

    # Preview
    console.print()
    console.print("[bold]Preview konversi (5 baris pertama):[/bold]")
    prev = Table(box=box.SIMPLE_HEAVY, header_style="bold magenta", show_lines=False)
    prev.add_column("ID",          style="cyan",  no_wrap=True, min_width=8)
    prev.add_column("url_filename", style="white", max_width=40)
    prev.add_column("→ Alt Text",   style="green", max_width=55)

    for row in valid_rows[:5]:
        fn  = str(row['url_filename']).strip()
        mid = str(row['id']).strip()
        prev.add_row(mid, fn, filename_to_alt(fn, domain))
    console.print(prev)

    # Confirm
    console.print()
    if not Confirm.ask("[bold yellow]Lanjutkan update ke WordPress?[/bold yellow]"):
        log_warn("Dibatalkan oleh user.")
        return

    console.print()

    # Proses update
    session = requests.Session()
    session.auth = (WP_USERNAME, WP_PASSWORD)

    sukses, gagal = 0, 0
    total = len(valid_rows)

    with Progress(
        SpinnerColumn(style="bold magenta"),
        TextColumn("[bold blue]{task.description}"),
        BarColumn(bar_width=35, style="magenta", complete_style="bold green"),
        TaskProgressColumn(),
        MofNCompleteColumn(),
        TimeElapsedColumn(),
        console=console,
    ) as progress:
        task = progress.add_task("Updating...", total=total)

        for i, row in enumerate(valid_rows, 1):
            fn  = str(row['url_filename']).strip()
            alt = filename_to_alt(fn, domain)

            try:
                media_id = int(float(str(row['id'])))
            except Exception:
                log_skip(f"[{i}/{total}] ID tidak valid: {row.get('id')}")
                gagal += 1
                progress.advance(task)
                continue

            ok = update_alt_text(session, api_url, media_id, alt)

            if ok:
                sukses += 1
                progress.update(task, advance=1,
                    description=f"[green]✔[/green] ID [bold]{media_id}[/bold] | {fn[:38]}")
            else:
                gagal += 1
                progress.update(task, advance=1,
                    description=f"[red]✘[/red] ID [bold]{media_id}[/bold] | {fn[:38]}")

            time.sleep(DELAY_SEC)

    # Summary
    console.print()
    console.print(Panel(
        f"[bold green]✔  Berhasil   : {sukses}[/bold green]\n"
        f"[bold red]✘  Gagal      : {gagal}[/bold red]\n"
        f"[bold yellow]⊘  Skip kosong: {skip_count}[/bold yellow]\n"
        f"──────────────────────\n"
        f"[bold]   Total      : {sukses + gagal + skip_count}[/bold]",
        title="[bold]📊 Hasil Update",
        border_style="green" if gagal == 0 else "yellow",
        padding=(0, 2),
    ))


# ============================================================
# CARA PAKAI:
#   1. Pastikan Colab Secrets sudah diset (pass_bali, admin_bali)
#   2. Jalankan cell ini
#   3. Upload file CSV hasil wp_media_fetcher.py
#   4. Preview muncul → konfirmasi → selesai!
# ============================================================

main()